# Criação do meta-Dataset para Stack Ensemble
Esse notebook tem como objetivo garantir a qualidade e a fidelidade dos dados para a fase do stack ensemble, ou seja, uma validação de integridade 


# Bibliotecas utilizadas

In [19]:
from pathlib import Path

import pandas as pd


## Configuracao Experimental

nessa cedula pegamos os csv's gerados pelo os 5 algoritmos e definimos: 
- Onde estão salvo o novo arquivo usado no stack ensemble 
- quais são os cinco modelos base que participarao do ensemble
- e quais colunas em conjunto identificam corretamente cada jogo


esse conjunto de colunas eu chamei de ``CHAVE_ESTRITA``, isso é so para garantir que o merge seja feito sobre o ``MESMO JOGO``, ou seja, evitar que probabilidades de partidas diferentes fossem unidas 

In [20]:
PASTA_DETALHADO = Path('..') / 'results' / 'experimento_03_detalhado'
SAIDA_STACK = Path('..') / 'results' / 'experimento_04_stack'
SAIDA_STACK.mkdir(parents=True, exist_ok=True)

ARQUIVOS = {
    'svm': 'svm_experimento_03_predicoes_detalhadas.csv',
    'knn': 'knn_experimento_03_predicoes_detalhadas.csv',
    'extra_trees': 'extra_trees_experimento_03_predicoes_detalhadas.csv',
    'regressao_logistica': 'regressao_logistica_experimento_03_predicoes_detalhadas.csv',
    'lightgbm': 'lightgbm_experimento_03_predicoes_detalhadas.csv',
}

CHAVE_ESTRITA = [
    'Janela Incremental',
    'Temporada',
    'Subpasta',
    'Teste Arquivo',
    'Posicao no Teste',
    'Data Jogo',
    'Round',
    'Ordem Jogo Temporada',
]

COLUNA_PROB = 'Probabilidade Classe 1'
COLUNA_ROTULO = 'Resultado Real'

print(f'Pasta de origem: {PASTA_DETALHADO.resolve()}')
print(f'Pasta de saida:  {SAIDA_STACK.resolve()}')
print(f'Modelos base:    {list(ARQUIVOS.keys())}')


Pasta de origem: C:\Users\Pedro conrado\Documents\stack-ensemble-prediction\results\experimento_03_detalhado
Pasta de saida:  C:\Users\Pedro conrado\Documents\stack-ensemble-prediction\results\experimento_04_stack
Modelos base:    ['svm', 'knn', 'extra_trees', 'regressao_logistica', 'lightgbm']


## Carregamento das bases de Predicao

Cada arquivo detalhado é carregado individualmente e armazenado em um dicionario indexado pelo nome do modelo


In [21]:
dfs = {}

for modelo, arquivo in ARQUIVOS.items():
    caminho = PASTA_DETALHADO / arquivo
    df = pd.read_csv(caminho)
    dfs[modelo] = df
    print(f"{modelo:22s} -> {len(df):6d} linhas | {df.shape[1]:2d} colunas")


svm                    ->  10740 linhas | 17 colunas
knn                    ->  10740 linhas | 17 colunas
extra_trees            ->  10740 linhas | 17 colunas
regressao_logistica    ->  10740 linhas | 17 colunas
lightgbm               ->  10740 linhas | 17 colunas


## Conferindo se cada jogo aparece apenas uma vez 

Antes de juntar tudo, estou conferindo se um mesmo jogo não aparece repetido dentro de um mesmo CSV, pois se isso acontecer, quando eu juntar tudo pode acabar duplicando linhas e gerar um conjunto final errado


In [22]:
ok_geral = True
for modelo, df in dfs.items():
    n_total = len(df)
    n_unicos = df[CHAVE_ESTRITA].drop_duplicates().shape[0]
    duplicatas = n_total - n_unicos
    status = 'OK' if duplicatas == 0 else f'ERRO - {duplicatas} duplicatas'

    if duplicatas > 0:
        ok_geral = False

    print(
        f"{modelo:22s} -> linhas={n_total:5d} | unicos={n_unicos:5d} | "
        f"duplicatas={duplicatas:3d} | {status}"
    )


svm                    -> linhas=10740 | unicos=10740 | duplicatas=  0 | OK
knn                    -> linhas=10740 | unicos=10740 | duplicatas=  0 | OK
extra_trees            -> linhas=10740 | unicos=10740 | duplicatas=  0 | OK
regressao_logistica    -> linhas=10740 | unicos=10740 | duplicatas=  0 | OK
lightgbm               -> linhas=10740 | unicos=10740 | duplicatas=  0 | OK


## Garantido se não tenho nenhuma coluna nula

O stacking depende diretamente de duas informacoes por observacao: o resultado final e a previsão dos algoritmos. Então eu estou antecipando o problema


In [23]:
ok_geral = True
for modelo, df in dfs.items():
    nulos_rotulo = df[COLUNA_ROTULO].isna().sum()
    nulos_prob = df[COLUNA_PROB].isna().sum()
    status = 'OK' if (nulos_rotulo + nulos_prob) == 0 else 'ATENCAO'

    if (nulos_rotulo + nulos_prob) > 0:
        ok_geral = False

    print(
        f"{modelo:22s} -> nulos rotulo={nulos_rotulo:3d} | "
        f"nulos prob={nulos_prob:3d} | {status}"
    )


svm                    -> nulos rotulo=  0 | nulos prob=  0 | OK
knn                    -> nulos rotulo=  0 | nulos prob=  0 | OK
extra_trees            -> nulos rotulo=  0 | nulos prob=  0 | OK
regressao_logistica    -> nulos rotulo=  0 | nulos prob=  0 | OK
lightgbm               -> nulos rotulo=  0 | nulos prob=  0 | OK


# Meta-Dataset


In [24]:
dfs_prob = []
for modelo, df in dfs.items():
    df_temp = df[CHAVE_ESTRITA + [COLUNA_ROTULO, COLUNA_PROB]].copy()
    df_temp = df_temp.rename(columns={COLUNA_PROB: f'prob_{modelo}'})
    dfs_prob.append(df_temp)

df_stack = dfs_prob[0]
for df_temp in dfs_prob[1:]:
    df_stack = df_stack.merge(
        df_temp.drop(columns=[COLUNA_ROTULO]),
        on=CHAVE_ESTRITA,
        how='inner'
    )

df_stack = df_stack[[
    'Janela Incremental',
    'Temporada',
    'Ordem Jogo Temporada',
    COLUNA_ROTULO,
    'prob_svm',
    'prob_knn',
    'prob_extra_trees',
    'prob_regressao_logistica',
    'prob_lightgbm',
]]

print(f'Shape: {df_stack.shape}')
print(f'Janelas:    {sorted(df_stack["Janela Incremental"].unique())}')
print(f'Temporadas: {sorted(df_stack["Temporada"].unique())}')
df_stack.head(3)

Shape: (10740, 9)
Janelas:    [np.int64(5), np.int64(10), np.int64(15)]
Temporadas: ['2008-2009', '2009-2010', '2011-2012', '2012-2013', '2013-2014', '2014-2015', '2015-2016', '2016-2017', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024']


,Janela Incremental,Temporada,Ordem Jogo Temporada,Resultado Real,prob_svm,prob_knn,prob_extra_trees,prob_regressao_logistica,prob_lightgbm
0,5,2008-2009,1,1,0.733335,0.666667,0.800000,0.799964,0.800000
1,5,2008-2009,2,0,0.769852,0.666667,0.833333,0.833331,0.833333
2,5,2008-2009,3,0,0.238587,0.333333,0.097143,0.535707,0.600000


## Salvando

In [25]:
caminho_saida = SAIDA_STACK / 'meta_dataset_stack.csv'
df_stack.to_csv(caminho_saida, index=False)

print(f'Salvo em: {caminho_saida.resolve()}')
print(f'Shape:    {df_stack.shape}')


Salvo em: C:\Users\Pedro conrado\Documents\stack-ensemble-prediction\results\experimento_04_stack\meta_dataset_stack.csv
Shape:    (10740, 9)
